In [2]:
# ============================================================
# Cell 1 : Imports & Global Configuration
# ============================================================

import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from docx import Document

warnings.filterwarnings("ignore")

# -------------------------------
# Paths
# -------------------------------

JD_PATH = "job_description.docx"

# -------------------------------
# Random seed
# -------------------------------

SEED = 42
np.random.seed(SEED)

# -------------------------------
# Skill Dictionaries
# -------------------------------

REQUIRED_SKILLS = {
    "python",
    "embeddings",
    "retrieval",
    "ranking",
    "vector database",
    "vector db",
    "faiss",
    "milvus",
    "pinecone",
    "weaviate",
    "qdrant",
    "elasticsearch",
    "opensearch",
    "evaluation",
    "ndcg",
    "mrr",
    "map"
}

PREFERRED_SKILLS = {
    "lora",
    "qlora",
    "peft",
    "llm",
    "fine tuning",
    "recommendation",
    "marketplace",
    "distributed systems",
    "opensource"
}

NEGATIVE_TERMS = {
    "computer vision",
    "speech",
    "robotics",
    "langchain",
    "research",
    "academic"
}

CONSULTING_COMPANIES = {
    "tcs",
    "infosys",
    "wipro",
    "accenture",
    "cognizant",
    "capgemini"
}

PRODUCT_COMPANIES = {
    "google",
    "meta",
    "amazon",
    "microsoft",
    "atlassian",
    "swiggy",
    "zomato",
    "razorpay",
    "flipkart"
}

print("Initialization Complete.")

Initialization Complete.


In [4]:
import re
from collections import defaultdict
from docx import Document

import spacy
from keybert import KeyBERT


nlp = spacy.load("en_core_web_lg")
kw_model = KeyBERT("all-MiniLM-L6-v2")


def read_docx(path):

    doc = Document(path)

    text = "\n".join(
        p.text.strip()
        for p in doc.paragraphs
        if p.text.strip()
    )

    return text


jd_text = read_docx(r"dataset/job_description.docx")

doc = nlp(jd_text)

sentences = [s.text.strip() for s in doc.sents]


entities = defaultdict(set)

for ent in doc.ents:

    entities[ent.label_].add(ent.text)


noun_chunks = sorted(
    set(chunk.text.strip() for chunk in doc.noun_chunks)
)


keywords = kw_model.extract_keywords(
    jd_text,
    top_n=100,
    keyphrase_ngram_range=(1,3),
    stop_words="english"
)

keywords = [
    {
        "keyword":k,
        "score":float(v)
    }
    for k,v in keywords
]

experience = None

m = re.search(
    r'(\d+)\s*[-–]\s*(\d+)\s*years',
    jd_text,
    flags=re.I
)

if m:

    experience = {

        "min":int(m.group(1)),
        "max":int(m.group(2))
    }

salary = re.findall(
    r'₹?\s*\d+\s*(?:LPA|lakhs?|crore)',
    jd_text,
    flags=re.I
)

notice = re.findall(
    r'(\d+)\s*day',
    jd_text,
    flags=re.I
)


work_mode = []

for mode in [
    "remote",
    "hybrid",
    "onsite",
    "flexible"
]:

    if mode.lower() in jd_text.lower():

        work_mode.append(mode)


locations = list(entities.get("GPE", []))



mandatory_patterns = [

    "must",
    "required",
    "need",
    "mandatory",
    "absolutely",
    "strong",
    "hands-on"

]

preferred_patterns = [

    "preferred",
    "nice to have",
    "good to have",
    "bonus",
    "like",
    "plus"

]

negative_patterns = [

    "do not",
    "don't",
    "not want",
    "won't",
    "avoid"

]

mandatory = []

preferred = []

negative = []

for s in sentences:

    ls = s.lower()

    if any(p in ls for p in mandatory_patterns):

        mandatory.append(s)

    elif any(p in ls for p in preferred_patterns):

        preferred.append(s)

    elif any(p in ls for p in negative_patterns):

        negative.append(s)



JD = {

    "raw_text":jd_text,

    "experience":experience,

    "salary":salary,

    "notice_period":notice,

    "locations":locations,

    "work_mode":work_mode,

    "entities":{

        k:list(v)

        for k,v in entities.items()

    },

    "keywords":keywords,

    "noun_phrases":noun_chunks,

    "mandatory_sentences":mandatory,

    "preferred_sentences":preferred,

    "negative_sentences":negative

}

print("="*80)
print("JD Parsed Successfully")
print("="*80)

print()

print("Experience")
print(JD["experience"])

print()

print("Locations")
print(JD["locations"])

print()

print("Work Mode")
print(JD["work_mode"])

print()

print("Top Keywords")

for k in JD["keywords"][:20]:

    print(k)

print()

print("Mandatory Sentences:",len(JD["mandatory_sentences"]))
print("Preferred Sentences :",len(JD["preferred_sentences"]))
print("Negative Sentences  :",len(JD["negative_sentences"]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4443.99it/s]


JD Parsed Successfully

Experience
{'min': 5, 'max': 9}

Locations
['Capgemini', 'India', 'Pune', 'Pinecone', 'Hyderabad', 'Noida', 'Mumbai']

Work Mode
['hybrid', 'flexible']

Top Keywords
{'keyword': 'senior ai engineer', 'score': 0.543}
{'keyword': 'ai engineer', 'score': 0.5309}
{'keyword': 'candidates hyderabad pune', 'score': 0.5167}
{'keyword': 'company redrob ai', 'score': 0.5042}
{'keyword': 'work ai experience', 'score': 0.4939}
{'keyword': 'ai engineering', 'score': 0.4867}
{'keyword': 'ai roles', 'score': 0.4858}
{'keyword': 'ai roles product', 'score': 0.4838}
{'keyword': 'ai engineering org', 'score': 0.4837}
{'keyword': 'ai native talent', 'score': 0.4802}
{'keyword': 'building new ai', 'score': 0.4724}
{'keyword': 'relocate noida pune', 'score': 0.4706}
{'keyword': 'new ai engineering', 'score': 0.4691}
{'keyword': 'career google', 'score': 0.4681}
{'keyword': 'ml ai roles', 'score': 0.4646}
{'keyword': 'relocation candidates tier', 'score': 0.4634}
{'keyword': 'pune ac

In [5]:

from collections import defaultdict
import networkx as nx
import re

CATEGORY_RULES = {

    "Technology": [
        "python","java","c++","sql","tensorflow","pytorch",
        "spark","kafka","docker","kubernetes",
        "aws","azure","gcp","git","linux",
        "milvus","pinecone","weaviate","faiss",
        "qdrant","elasticsearch","opensearch"
    ],

    "Domain":[
        "machine learning","deep learning","nlp",
        "computer vision","recommendation",
        "retrieval","ranking","search",
        "backend","frontend","devops",
        "marketing","finance","healthcare",
        "robotics","speech"
    ],

    "Responsibility":[
        "design","develop","deploy","build",
        "maintain","lead","mentor","optimize",
        "evaluate","research","implement",
        "ship","architect","monitor"
    ],

    "SoftSkill":[
        "communication","ownership",
        "leadership","teamwork",
        "problem solving",
        "collaboration"
    ]
}

G = nx.DiGraph()


G.add_node(
    "JOB",
    type="ROOT"
)

if JD["experience"]:

    G.add_node(
        "Experience",
        type="Constraint",
        value=JD["experience"]
    )

    G.add_edge(
        "JOB",
        "Experience",
        relation="requires"
    )


for mode in JD["work_mode"]:

    G.add_node(
        mode,
        type="WorkMode"
    )

    G.add_edge(
        "JOB",
        mode,
        relation="prefers"
    )


for loc in JD["locations"]:

    G.add_node(
        loc,
        type="Location"
    )

    G.add_edge(
        "JOB",
        loc,
        relation="located_at"
    )


classified = defaultdict(list)

for item in JD["keywords"]:

    keyword = item["keyword"].lower()

    found = False

    for category, vocab in CATEGORY_RULES.items():

        if keyword in vocab:

            classified[category].append(item)

            G.add_node(
                keyword,
                type=category,
                score=item["score"]
            )

            G.add_edge(
                "JOB",
                keyword,
                relation="mentions"
            )

            found = True
            break

    if not found:

        G.add_node(
            keyword,
            type="Unknown",
            score=item["score"]
        )

        G.add_edge(
            "JOB",
            keyword,
            relation="mentions")


for idx,s in enumerate(JD["mandatory_sentences"]):

    node = f"Mandatory_{idx}"

    G.add_node(
        node,
        type="Mandatory",
        text=s
    )

    G.add_edge(
        "JOB",
        node,
        relation="requires"
    )


for idx,s in enumerate(JD["preferred_sentences"]):

    node = f"Preferred_{idx}"

    G.add_node(
        node,
        type="Preferred",
        text=s
    )

    G.add_edge(
        "JOB",
        node,
        relation="prefers"
    )


for idx,s in enumerate(JD["negative_sentences"]):

    node = f"Negative_{idx}"

    G.add_node(
        node,
        type="Negative",
        text=s
    )

    G.add_edge(
        "JOB",
        node,
        relation="avoids"
    )


print("="*80)
print("Knowledge Graph Created")
print("="*80)

print()

print("Nodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())

print()

print("Categories")

for cat,items in classified.items():

    print(f"{cat:<18} {len(items)}")

Knowledge Graph Created

Nodes : 132
Edges : 131

Categories


In [6]:

from sentence_transformers import SentenceTransformer
import numpy as np
import re


MODEL_NAME = "BAAI/bge-large-en-v1.5"

embedder = SentenceTransformer(MODEL_NAME)

print("Embedding Model Loaded")

requirement_sentences = []

requirement_sentences.extend(JD["mandatory_sentences"])
requirement_sentences.extend(JD["preferred_sentences"])
requirement_sentences.extend(JD["negative_sentences"])

# Remove duplicates
requirement_sentences = list(dict.fromkeys(requirement_sentences))


requirement_embeddings = embedder.encode(
    requirement_sentences,
    normalize_embeddings=True,
    show_progress_bar=True
)


JD_REQUIREMENTS = []

for sentence, embedding in zip(
    requirement_sentences,
    requirement_embeddings
):

    s = sentence.lower()

    if sentence in JD["mandatory_sentences"]:
        importance = "mandatory"
        weight = 1.0

    elif sentence in JD["preferred_sentences"]:
        importance = "preferred"
        weight = 0.7

    else:
        importance = "negative"
        weight = -1.0

    JD_REQUIREMENTS.append({

        "text": sentence,

        "importance": importance,

        "weight": weight,

        "embedding": embedding

    })

print()

print("="*80)
print("Requirement Objects Created")
print("="*80)

print("Total:", len(JD_REQUIREMENTS))

print()

for r in JD_REQUIREMENTS[:5]:

    print("-----------------------------------------")
    print(r["importance"].upper())
    print(r["text"][:150])

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4872.81it/s]


Embedding Model Loaded


Batches: 100%|██████████| 1/1 [00:06<00:00,  6.99s/it]


Requirement Objects Created
Total: 21

-----------------------------------------
MANDATORY
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (
-----------------------------------------
MANDATORY
So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
-----------------------------------------
MANDATORY
We need someone who is simultaneously comfortable with two things that sound contradictory:
Deep technical depth in modern ML systems — embeddings, re
-----------------------------------------
MANDATORY
Scrappy product-engineering attitude — willing to ship a working ranker in a week even if the underlying ML is "obviously suboptimal," because we need
-----------------------------------------
MANDATORY
We need both modes available in the same person, and we'd rather you tilt slightly toward shipper than toward res

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

FIELDS = [
    ("headline", lambda c: c["profile"].get("headline", "")),
    ("summary", lambda c: c["profile"].get("summary", "")),
    (
        "career",
        lambda c: " ".join(
            x.get("description", "")
            for x in c.get("career_history", [])
        ),
    ),
    (
        "skills",
        lambda c: " ".join(
            x.get("name", "")
            for x in c.get("skills", [])
        ),
    ),
]

def build_candidate_representation(candidate):

    obj = {}

    obj["candidate_id"] = candidate["candidate_id"]

    obj["experience"] = candidate["profile"].get(
        "years_of_experience", 0
    )

    obj["title"] = candidate["profile"].get(
        "current_title", ""
    )

    obj["company"] = candidate["profile"].get(
        "current_company", ""
    )

    obj["industry"] = candidate["profile"].get(
        "current_industry", ""
    )

    obj["location"] = candidate["profile"].get(
        "location", ""
    )

    obj["signals"] = candidate["redrob_signals"]

    obj["text"] = {}

    obj["embeddings"] = {}
    
    obj["skills"] = candidate.get("skills", [])

    obj["career_history"] = candidate.get("career_history", [])

    obj["education"] = candidate.get("education", [])

    obj["profile"] = candidate.get("profile", {})

    for name, extractor in FIELDS:

        txt = extractor(candidate).strip()

        obj["text"][name] = txt

        if len(txt):

            emb = embedder.encode(
                txt,
                normalize_embeddings=True
            )

        else:

            emb = np.zeros(
                embedder.get_sentence_embedding_dimension()
            )

        obj["embeddings"][name] = emb

    return obj


import json

with open(r"dataset/sample_candidates.json", "r", encoding="utf-8") as f:
    sample_candidates = json.load(f)

candidate = sample_candidates[0]

candidate_obj = build_candidate_representation(candidate)

candidate_obj.keys()

dict_keys(['candidate_id', 'experience', 'title', 'company', 'industry', 'location', 'signals', 'text', 'embeddings', 'skills', 'career_history', 'education', 'profile'])

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

CANDIDATE_FIELDS = [
    "headline",
    "summary",
    "career",
    "skills"
]

def cosine(a, b):

    return float(
        cosine_similarity(
            a.reshape(1, -1),
            b.reshape(1, -1)
        )[0][0]
    )


def build_similarity_matrix(candidate):

    matrix = []

    for req in JD_REQUIREMENTS:

        row = []

        for field in CANDIDATE_FIELDS:

            row.append(
                cosine(
                    req["embedding"],
                    candidate["embeddings"][field]
                )
            )

        matrix.append(row)

    return np.array(matrix)

In [9]:
import numpy as np

def aggregate_similarity(matrix):

    row_best = matrix.max(axis=1)

    col_best = matrix.max(axis=0)

    return {

        "req_mean": row_best.mean(),

        "req_max": row_best.max(),

        "req_min": row_best.min(),

        "req_std": row_best.std(),

        "headline_match": col_best[0],

        "summary_match": col_best[1],

        "career_match": col_best[2],

        "skill_match": col_best[3]

    }

In [10]:
import numpy as np

def experience_score(candidate_exp, jd_exp):

    if jd_exp is None:
        return 1.0

    mn = jd_exp["min"]
    mx = jd_exp["max"]

    if mn <= candidate_exp <= mx:
        return 1.0

    if candidate_exp < mn:
        return max(0, 1 - (mn - candidate_exp) / mn)

    return max(0, 1 - (candidate_exp - mx) / mx)


def behavior_score(signals):

    features = [

        signals["profile_completeness_score"] / 100,

        signals["recruiter_response_rate"],

        signals["interview_completion_rate"],

        min(signals["search_appearance_30d"], 200) / 200,

        min(signals["saved_by_recruiters_30d"], 20) / 20,

        min(signals["connection_count"], 500) / 500,

        0 if signals["github_activity_score"] == -1
        else signals["github_activity_score"] / 100,

        1 if signals["open_to_work_flag"] else 0,

        1 if signals["verified_email"] else 0,

        1 if signals["verified_phone"] else 0,

        1 if signals["linkedin_connected"] else 0

    ]

    return np.mean(features)


def penalty_score(candidate):

    penalty = 0

    if candidate["signals"]["notice_period_days"] > 90:
        penalty += 0.15

    if candidate["signals"]["avg_response_time_hours"] > 168:
        penalty += 0.10

    if candidate["signals"]["recruiter_response_rate"] < 0.10:
        penalty += 0.15

    return penalty


def score_candidate(candidate):

    matrix = build_similarity_matrix(candidate)

    semantic = aggregate_similarity(matrix)

    experience = experience_score(
        candidate["experience"],
        JD["experience"]
    )

    behavior = behavior_score(
        candidate["signals"]
    )

    penalty = penalty_score(candidate)

    scores = {

        "semantic": semantic,

        "experience": experience,

        "behavior": behavior,

        "penalty": penalty

    }

    return scores

In [11]:
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_features=5000
)

tfidf.fit([JD["raw_text"]])

jd_tfidf = tfidf.transform([JD["raw_text"]])


def exact_skill_overlap(candidate):

    jd_skills = set()

    for kw in JD["keywords"]:
        jd_skills.add(
            kw["keyword"].lower().strip()
        )

    cand_skills = set(
        s["name"].lower().strip()
        for s in candidate["skills"]
    )

    if len(jd_skills) == 0:
        return 0

    return len(
        jd_skills.intersection(cand_skills)
    ) / len(jd_skills)


def fuzzy_skill_overlap(candidate):

    candidate_skills = [
        s["name"].lower()
        for s in candidate["skills"]
    ]

    jd_keywords = [
        k["keyword"].lower()
        for k in JD["keywords"]
    ]

    scores = []

    for jd_skill in jd_keywords:

        best = 0

        for skill in candidate_skills:

            best = max(
                best,
                fuzz.token_sort_ratio(
                    jd_skill,
                    skill
                ) / 100
            )

        scores.append(best)

    if len(scores) == 0:
        return 0

    return np.mean(scores)


def tfidf_similarity(candidate):

    txt = " ".join([

        candidate["text"]["headline"],

        candidate["text"]["summary"],

        candidate["text"]["career"],

        candidate["text"]["skills"]

    ])

    cand_vec = tfidf.transform([txt])

    return float(

        cosine_similarity(
            jd_tfidf,
            cand_vec
        )[0][0]

    )


def semantic_similarity(candidate):

    matrix = build_similarity_matrix(candidate)

    features = aggregate_similarity(matrix)

    return features["req_mean"]


def hybrid_similarity(candidate):

    return {

        "semantic": semantic_similarity(candidate),

        "tfidf": tfidf_similarity(candidate),

        "exact_skill": exact_skill_overlap(candidate),

        "fuzzy_skill": fuzzy_skill_overlap(candidate)

    }

In [12]:
WEIGHTS = {

    "semantic": 0.40,

    "tfidf": 0.10,

    "exact_skill": 0.10,

    "fuzzy_skill": 0.05,

    "experience": 0.10,

    "behavior": 0.20,

    "penalty": 0.05

}


def final_score(candidate):

    sim = hybrid_similarity(candidate)

    exp = experience_score(
        candidate["experience"],
        JD["experience"]
    )

    beh = behavior_score(
        candidate["signals"]
    )

    pen = penalty_score(candidate)

    score = (

        WEIGHTS["semantic"] * sim["semantic"] +

        WEIGHTS["tfidf"] * sim["tfidf"] +

        WEIGHTS["exact_skill"] * sim["exact_skill"] +

        WEIGHTS["fuzzy_skill"] * sim["fuzzy_skill"] +

        WEIGHTS["experience"] * exp +

        WEIGHTS["behavior"] * beh -

        WEIGHTS["penalty"] * pen

    )

    return float(score)


def generate_reason(candidate, score):

    reasons = []

    if candidate["experience"] >= JD["experience"]["min"]:
        reasons.append(
            f'{candidate["experience"]:.1f} years experience'
        )

    if score > 0.80:
        reasons.append("strong semantic match")

    elif score > 0.65:
        reasons.append("good semantic alignment")

    if candidate["signals"]["github_activity_score"] > 60:
        reasons.append("active GitHub profile")

    if candidate["signals"]["open_to_work_flag"]:
        reasons.append("open to work")

    if candidate["signals"]["notice_period_days"] <= 30:
        reasons.append("short notice period")

    if len(reasons) == 0:
        reasons.append("partial profile match")

    return ". ".join(reasons[:3]) + "."


def rank_candidates(candidates):

    rows = []

    for cand in candidates:

        obj = build_candidate_representation(cand)

        score = final_score(obj)

        rows.append({

            "candidate_id": cand["candidate_id"],

            "score": score,

            "reasoning": generate_reason(obj, score)

        })

    rows = sorted(
        rows,
        key=lambda x: (
            -x["score"],
            x["candidate_id"]
        )
    )

    rows = rows[:100]

    for i, row in enumerate(rows):

        row["rank"] = i + 1

    return rows

In [16]:
import json
import gzip
import pandas as pd
from tqdm.auto import tqdm

INPUT_FILE = r"dataset/candidates.jsonl"     
OUTPUT_FILE = "submission.csv"

USE_GZIP = INPUT_FILE.endswith(".gz")

rows = []

if USE_GZIP:
    reader = gzip.open(INPUT_FILE, "rt", encoding="utf-8")
else:
    reader = open(INPUT_FILE, "r", encoding="utf-8")

with reader as f:

    for line in tqdm(f):

        candidate = json.loads(line)

        candidate_obj = build_candidate_representation(candidate)

        sim = hybrid_similarity(candidate_obj)

        exp = experience_score(
            candidate_obj["experience"],
            JD["experience"]
        )

        beh = behavior_score(
            candidate_obj["signals"]
        )

        pen = penalty_score(
            candidate_obj
        )

        score = (

            WEIGHTS["semantic"] * sim["semantic"]

            + WEIGHTS["tfidf"] * sim["tfidf"]

            + WEIGHTS["exact_skill"] * sim["exact_skill"]

            + WEIGHTS["fuzzy_skill"] * sim["fuzzy_skill"]

            + WEIGHTS["experience"] * exp

            + WEIGHTS["behavior"] * beh

            - WEIGHTS["penalty"] * pen

        )

        reasoning = generate_reason(
            candidate_obj,
            score
        )

        rows.append({

            "candidate_id": candidate["candidate_id"],

            "score": float(score),

            "reasoning": reasoning

        })

submission = pd.DataFrame(rows)

submission = submission.sort_values(

    ["score", "candidate_id"],

    ascending=[False, True]

).reset_index(drop=True)

submission = submission.head(100)

submission["rank"] = range(1, 101)

submission = submission[

    [

        "candidate_id",

        "rank",

        "score",

        "reasoning"

    ]

]

submission.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8"

)

print(submission.head())

print()

print("Saved:", OUTPUT_FILE)

2482it [1:10:13,  1.70s/it]


KeyboardInterrupt: 